In [3]:
import re
from pathlib import Path

from src.indexer.scan_dir import iter_file_metadata

home = Path("/home/sigma.sbrf.ru@16525722")

_inpox_path = re.compile(r"/ImapMail/imap.sberbank.ru/INBOX$")
_db_path = re.compile(r"global-messages-db.sqlite")


def _filter_inbox(path: str):
    # print(path)
    return _inpox_path.search(path) is not None


def _filter_db(path: str):
    return _inpox_path.search(path) is not None


for f in iter_file_metadata(
    str(home / ".thunderbird"),
    exclude_patt=None,
    filter_func=_filter_inbox,
    check_parsers=False,
):
    print(f)
    continue

FileMetadata(path='/home/sigma.sbrf.ru@16525722/.thunderbird/1nvbyfwy.default-default/ImapMail/imap.sberbank.ru/INBOX', size=5809580017, mtime_seconds=1764340254)


In [2]:
from datetime import datetime

from src.thunderbird.read_messages import read_messages

p = str(Path(f.path).parent.parent.parent / "global-messages-db.sqlite")
f"{p}?mode=ro&immutable=1"


next(
    read_messages(
        db_path=f"file:{p}",
        min_date=datetime(2023, 12, 1),
        split_threads=False,
        # progress=prog,
    )
)

OperationalError: database is locked

In [ ]:
# msg, (d, start, stop) = next(_iter_emails(f.path))

In [ ]:
with open(f.path, "br") as f:
    f.seek(start)
    data = f.read(stop - start)

In [ ]:
with open("data/sample.eml", "bw") as f:
    f.write(data)

In [ ]:
_decode_filename(msg["Thread-Topic"])

'Роскошный раг: прокси до ГЧ'

In [ ]:
import re
from collections import defaultdict
from email.message import Message

import networkx as nx
from bs4 import BeautifulSoup
from tqdm import tqdm

from src.thunderbird.lazy_iter_mbox import _iter_emails
from src.thunderbird.read_messages import extract_unique_body

_patt_clean = re.compile(r"\n{2,}", re.MULTILINE | re.DOTALL)
# _patt_split


def get_charset(body: str):
    soup = BeautifulSoup(body)
    if soup.head and soup.head.find("meta", charset=True):
        charset_value = soup.head.find("meta", charset=True)["charset"]
        return charset_value


def _get_part(part: Message) -> str | None:
    ctype = part.get_content_type()
    cdispo = str(part.get("Content-Disposition") or "")
    if (
        ctype
        in {
            "text/plain",
            "text/html",
            # "multipart/alternative",
        }
        and "attachment" not in cdispo
    ):
        charset = part.get_content_charset()
        # print("Charset: ", charset)
        # print("CType: ", ctype)
        body: str = part.get_payload(decode=True).decode(encoding=charset)  # type: ignore
        if ctype == "text/html":
            body = BeautifulSoup(body).get_text()
        if body is not None:
            body = _patt_clean.sub("\n\n", body.strip())
            body = extract_unique_body(body).strip()
        return body


def get_body(message: Message, encoding: str = "utf-8") -> str | None:
    # print(
    #     "-" * 50,
    #     "\nSubject: ",
    #     _decode_filename(message["Subject"]),
    #     "\n",
    #     "Thread-Index: ",
    #     message["Thread-Index"],
    #     "\n",
    #     "-" * 50,
    #     sep="",
    # )
    if message.is_multipart():
        for part in message.walk():
            if body := _get_part(part):
                return body
    elif body := _get_part(message):
        return body


sent = "/home/sigma.sbrf.ru@16525722/.thunderbird/1nvbyfwy.default-default/ImapMail/imap.sberbank.ru/&BB4EQgQ,BEAEMAQyBDsENQQ9BD0ESwQ1-"
inbox = "/home/sigma.sbrf.ru@16525722/.thunderbird/1nvbyfwy.default-default/ImapMail/imap.sberbank.ru/INBOX"

ret = defaultdict(list)
graph = nx.Graph()
for pp in [sent, inbox]:
    for i, (msg, (d, start, stop)) in tqdm(enumerate(_iter_emails(pp))):
        print(i, end="\r")
        if thId := msg["Thread-Index"]:
            body = get_body(msg)
            ret[thId].append((d, (body, msg.keys())))
        if (reply_to := msg["In-Reply-To"]) and (msg_id := msg["Message-ID"]):
            graph.add_edge(msg_id, reply_to)
            # graph[msg_id].add(reply_to)
        if i >= 100:
            break
nx.wrigr

KeyboardInterrupt: 

In [ ]:
import pickle

with open("data/graph.pickle", "br") as f:
    graph = pickle.load(f)

In [8]:
import networkx as nx

list(nx.connected_components(graph))

[{'<008c78f3-5f39-446b-a6e8-bda63acdabc8@sberbank.ru>',
  '<C75101BE-4FEC-4C10-9165-A83C20DC8F18@sberbank.ru>',
  '<c667cf65-4e88-4fdd-83ba-9983afaa3e98@sberbank.ru>',
  '<fc53f97c-fa1c-4556-885f-57233564da43@sberbank.ru>'},
 {'<008901dc0c55$5df8eb60$19eac220$@omega.sbrf.ru>',
  '<5d0bded6-ff9e-4af8-bb4b-aec69e506e0a@sberbank.ru>'},
 {'<02ba1990-801d-4ffc-9e21-ff5daaf4770d@sberbank.ru>',
  '<04718e8d-5fdb-42a0-8739-ee73c64b9b36@sberbank.ru>',
  '<06f27aee0f4d44b88359f647e62a467d@CAB-VSP-MBX5041.sigma.sbrf.ru>',
  '<0dec56bc-5a06-49e9-958c-2a79406d7b4d@sberbank.ru>',
  '<0e7a38ef-0614-4c2d-859f-95b659856d86@sberbank.ru>',
  '<116ff6e5-a8a3-4c3c-96f9-a77d1ce26c73@sberbank.ru>',
  '<1262e129-5a85-4bd3-b1cc-398dab1cb97f@sberbank.ru>',
  '<131682574.7.1763387312257@sberbank.ru>',
  '<176e76680544424aa418198b76a39689@sberbank.ru>',
  '<215927003.3.1763386423090@sberbank.ru>',
  '<22d04884-6b38-4ae5-90c6-ac20d5c99981@sberbank.ru>',
  '<2517f8a5-eb7c-4398-8a1b-43a3604665b0@sberbank.ru>',
  '<2

In [ ]:
from time import sleep

from rich.progress import track

for t in track(range(10)):
    sleep(1)
    # print(
    #     t,
    #     end="\r",
    # )

Output()

In [3]:
from time import sleep

# from tqdm.notebook import trange


for t in range(10):
    sleep(1)
    print(
        t,
        end="\r",
    )

In [9]:
keys_all = set()
for k, v in ret.items():
    # keys_all |= set(keys)
    for d, (vv, keys) in v:
        keys_all |= set(keys)
        # print("-" * 50)
        # print(d)
        # print(vv)

    # if len(v) > 1:
    #     print('=' * 50)
keys_all

{'Accept-Language',
 'CC',
 'Content-ID',
 'Content-Language',
 'Content-Transfer-Encoding',
 'Content-Type',
 'Date',
 'From',
 'Importance',
 'In-Reply-To',
 'MIME-Version',
 'Message-ID',
 'Received',
 'References',
 'Reply-To',
 'Return-Path',
 'Sender',
 'Subject',
 'Thread-Index',
 'Thread-Topic',
 'To',
 'X-Auto-Response-Suppress',
 'X-KSE-AntiSpam-Interceptor-Info',
 'X-KSE-Antispam-Frontend-Serialized-Headers',
 'X-KSE-Antivirus-Info',
 'X-KSE-Antivirus-Interceptor-Info',
 'X-KSE-Attachment-Filter-Triggered-Filters',
 'X-KSE-Attachment-Filter-Triggered-Rules',
 'X-KSE-BulkMessagesFiltering-Scan-Result',
 'X-KSE-ServerInfo',
 'X-MS-Exchange-Inbox-Rules-Loop',
 'X-MS-Exchange-MessageSentRepresentingType',
 'X-MS-Exchange-Organization-AVStamp-Mailbox',
 'X-MS-Exchange-Organization-AuthAs',
 'X-MS-Exchange-Organization-AuthMechanism',
 'X-MS-Exchange-Organization-AuthSource',
 'X-MS-Exchange-Organization-Network-Message-Id',
 'X-MS-Exchange-Organization-RecordReviewCfmType',
 'X-M